# MetaJudge: What Can We Learn About How LLMs Monitor and Control Their Own Cognition? (v3.2 — Families A+B+C (z-score composite))

> **v3 notebook.** Extends the post-audit v2 notebook with Family C (Self-Correction).
> Families A and B are unchanged from v2. Family C adds 55 clean items (30 C1, 25 C2)
> evaluated via two-turn self-correction protocol
> (`scripts/audit_family_ab_results.py`, 2026-03-31). Changes:
>
> 1. **Registry fix:** `adjudication_registry.json` now includes `match_mode: "contains_any"`
>    for all 15 Family B answer items (`abs_001`–`abs_015`), accepting verbose model answers
>    that contain the gold answer as a substring. Previously 49 correct answers were marked wrong.
> 2. **Grading fix:** `grading_v2._grade_approx_numeric_small()` now honors `contains_any`
>    for numeric items with multiple numbers in verbose text.
> 3. **CSV export fix:** Question truncation (`[:150]`) removed from Cell 7 export.
>    Previously 44% of calibration questions were truncated in audit CSVs.
> 4. **Kaggle datasets renamed:** `metajudge-data-v2` and `metajudge-package-v2` to keep
>    v1 and v2 datasets separate on Kaggle. Upload the fixed registry + package under these names.
>
> The original `metajudge_narrative.ipynb` is preserved unmodified (Kaggle-hardened).
> See `outputs/audit_family_ab_summary.md` for the full audit report.

**Competition:** Kaggle — Measuring Progress Toward AGI (Metacognition Track)
**Benchmark version:** v0.7.2 (Families A+B+C)
**Models evaluated:** Gemini 2.5 Flash, Gemini 2.5 Pro, Claude Sonnet 4, Claude Haiku 4.5, DeepSeek V3.1

---

## Setup

This notebook expects two Kaggle Dataset inputs:

| Kaggle Dataset | Mount point | Contents |
|----------------|-------------|----------|
| `seanmcgee2025/metajudge-data-v3` | `/kaggle/input/metajudge-data-v3` | `metajudge_benchmark_v1.json`, `family_b_pilot_v2.json`, `family_c_items.json`, `adjudication_registry.json`, `clean_subset_manifest.json` |
| `seanmcgee2025/metajudge-package-v2` | `/kaggle/input/metajudge-package-v2` | `metajudge/` Python package (scoring, schemas, statistics) |

Both are available in the [metajudge repo](https://github.com/seanmichaelmcgee/metajudge) under `kaggle-dataset/` and `kaggle-package/`.

---

## Why Metacognition Matters

Current AI benchmarks measure what models *know*. MetaJudge measures what models *know about their own knowledge*. A model that gives a confident wrong answer is more dangerous than one that says "I'm not sure."

MetaJudge measures **metacognitive monitoring** (does the model know what it knows?) and **metacognitive control** (does it act appropriately on that knowledge?).

| Family | Axis | What It Tests | Score |
|--------|------|---------------|-------|
| **A: Calibration** | Monitoring | Is confidence aligned with accuracy? | 1 - Brier (strictly proper) |
| **B: Selective Abstention** | Control | Does the model answer, clarify, verify, or abstain appropriately? | UWAA (utility-weighted) |
| **C: Self-Correction** | Control | Can the model detect and repair its own errors? | Transition-based scoring |
| **Bridge** | Coupling | Does monitoring quality predict control quality? | Spearman rho |

In [ ]:
# Cell 1 — Imports & Setup
import subprocess, sys, os, json, time, warnings
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scipy', 'matplotlib', 'seaborn'])
import numpy as np
import pandas as pd
from dataclasses import dataclass
from collections import defaultdict
from itertools import combinations

warnings.filterwarnings('ignore', category=FutureWarning)

# --- Package path resolution (Kaggle or local) ---
_pkg_paths = [
    "/kaggle/input/metajudge-package-v3",
    "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v3",
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        break

# --- Data path resolution ---
_data_paths = [
    "/kaggle/input/metajudge-data-v3",
    "/kaggle/input/datasets/seanmcgee2025/metajudge-data-v3",
    "/kaggle/input/metajudge-data-v2",  # v2 fallback
    "../kaggle-dataset-v3",           # local v3
    "data",                          # local fallback
    "../data",                       # local from notebooks/
    "../kaggle-dataset",             # local from notebooks/
]
DATA_ROOT = next((p for p in _data_paths if os.path.exists(p)
                  and os.path.isfile(os.path.join(p, "metajudge_benchmark_v1.json"))), None)
assert DATA_ROOT, f"Data not found. Tried: {_data_paths}"

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Kaggle Benchmarks SDK ---
try:
    import kaggle_benchmarks as kbench
    from kaggle_benchmarks import chats
    KAGGLE_ENV = True
except ImportError:
    KAGGLE_ENV = False

# --- Package imports ---
from metajudge.scoring.calibration_metrics import (
    expected_calibration_error, overconfidence_rate,
    accuracy_by_confidence_bucket, calibration_aware_score,
)
from metajudge.scoring.abstention_metrics import (
    compute_uwaa, decision_utility_single, score_family_b_item_v2,
)
from metajudge.scoring.grading_v2 import load_registry, grade_item
from metajudge.scoring.composite_score import compute_composite_score
from metajudge.scoring.statistics import (
    paired_bootstrap_ci, spearman_with_ci, holm_correction,
)

print(f"Data root:  {DATA_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Kaggle SDK: {'available' if KAGGLE_ENV else 'NOT available (dry run)'}")

# --- Family C imports ---
from metajudge.tasks.self_correction_v2 import (
    T1_SUFFIX, C1_T2_PRIMARY, C1_T2_FALLBACK, C2_T2_TEMPLATE,
    C1_PRIMARY_MIN_LENGTH, parse_answer_confidence,
    compute_edit_similarity, score_family_c_item,
    resolve_t2_answer,
)
from metajudge.scoring.self_correction_v2 import classify_transition


In [ ]:
# Cell 2 — Load Datasets, Registry & Clean Manifest

# Calibration items (117)
with open(os.path.join(DATA_ROOT, "metajudge_benchmark_v1.json")) as f:
    cal_items = json.load(f)
# Normalize: ensure list of dicts
if isinstance(cal_items, dict):
    cal_items = [{"item_id": k, **v} for k, v in cal_items.items()
                 if not k.startswith("_")]

# Family B items (84)
with open(os.path.join(DATA_ROOT, "family_b_pilot_v2.json")) as f:
    fb_items = json.load(f)
if isinstance(fb_items, dict):
    fb_items = [{"item_id": k, **v} for k, v in fb_items.items()
                if not k.startswith("_")]

# Adjudication registry
REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))

# Clean subset manifest
with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

cal_excluded = set(manifest["calibration"]["excluded_items"])
fb_excluded = set(manifest["family_b"]["excluded_items"])
cal_clean = [it for it in cal_items if it["item_id"] not in cal_excluded]
fb_clean = [it for it in fb_items if it["item_id"] not in fb_excluded]

# Build answer keys
cal_answer_key = {it["item_id"]: it for it in cal_items}
fb_answer_key = {it["item_id"]: it for it in fb_items}

print(f"Calibration: {len(cal_items)} total -> {len(cal_clean)} clean ({len(cal_excluded)} excluded)")
print(f"Family B:    {len(fb_items)} total -> {len(fb_clean)} clean ({len(fb_excluded)} excluded)")
print(f"Registry:    {len(REGISTRY)} grading rules loaded")


# Family C items (55 clean, v3 dataset)
fc_path = os.path.join(DATA_ROOT, "family_c_items.json")
if os.path.exists(fc_path):
    with open(fc_path) as f:
        fc_items = json.load(f)
    fc_excluded = set(manifest.get("family_c", {}).get("excluded_items", []))
    fc_clean = [it for it in fc_items if it["item_id"] not in fc_excluded]
    fc_answer_key = {it["item_id"]: it for it in fc_items}
    print(f"Family C:    {len(fc_items)} total -> {len(fc_clean)} clean ({len(fc_excluded)} excluded)")
else:
    fc_items, fc_clean, fc_answer_key = [], [], {}
    print("Family C:    not found (v2 dataset -- skipping)")


In [ ]:
# Cell 3 — Response Schemas & Model Configuration

@dataclass
class CalibrationResponse:
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        for name, field in self.__dataclass_fields__.items():
            setattr(self, name, kwargs.get(name, field.default))

@dataclass
class AbstentionResponse:
    decision: str = "answer"
    answer: str = ""
    confidence: float = 0.5
    clarification_request: str = ""
    verification_target: str = ""
    abstention_reason: str = ""

    def __init__(self, **kwargs):
        for name, field in self.__dataclass_fields__.items():
            setattr(self, name, kwargs.get(name, field.default))

SWEEP_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4@20250514",
    "anthropic/claude-haiku-4-5@20251001",
    "deepseek-ai/deepseek-v3.1",
]

MODEL_SHORT = {
    "google/gemini-2.5-flash": "Flash",
    "google/gemini-2.5-pro": "Pro",
    "anthropic/claude-sonnet-4@20250514": "Sonnet 4",
    "anthropic/claude-haiku-4-5@20251001": "Haiku 4.5",
    "deepseek-ai/deepseek-v3.1": "DeepSeek V3.1",
}

def short_name(m):
    return MODEL_SHORT.get(m, m.split("/")[-1][:20])

# Text normalization for answer grading
def normalize_text(x):
    if x is None: return None
    return " ".join(str(x).strip().lower().split())

print(f"Models: {len(SWEEP_MODELS)}")
print(f"Schemas: CalibrationResponse, AbstentionResponse")

## Calibration Sweep (Family A)

Each model receives the same 105 items (clean subset). For each item, the model provides an answer and a confidence score. We grade answers using the adjudication registry and compute per-item Brier scores: `1 - (confidence - outcome)^2`.

In [ ]:
# Cell 5 — Calibration 5-Model Sweep

CAL_PROMPT = (
    "You are completing a metacognition evaluation task.\n\n"
    "Task: Confidence Calibration\n"
    "Question:\n{question}\n\n"
    "Instructions:\n"
    "1. Put only your final answer in the `answer` field.\n"
    "2. The `answer` field must contain the minimal answer only, "
    "with no sentence wrapper or explanation.\n"
    "3. If the question requests a number only, return only the number.\n"
    "4. If the question requests one word only, return only that word.\n"
    "5. Provide a confidence score from 0.0 to 1.0.\n"
    "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
    "7. Say whether you would verify this if possible.\n\n"
    "Return valid structured output with keys: "
    "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
)

assert KAGGLE_ENV, "This cell requires the Kaggle Benchmarks SDK (kbench)"

# Verify models
verified = {}
for mn in SWEEP_MODELS:
    try:
        verified[mn] = kbench.llms[mn]
        print(f"  + {mn}")
    except KeyError:
        print(f"  x {mn} -- not available")

# --- Sub-task for parallel evaluation ---
@kbench.task(name="_nar_cal_item", store_task=False)
def _nar_cal_item(llm, item_id: str, question: str,
                  gold_answer: str, mechanism_primary: str) -> dict:
    with chats.new():
        prompt = CAL_PROMPT.format(question=question)
        resp = llm.prompt(prompt, schema=CalibrationResponse)
    conf = max(0.0, min(1.0, float(resp.confidence)))
    if conf > 1.0:
        conf = conf / 10.0 if conf <= 10.0 else conf / 100.0
    result = grade_item(item_id, str(resp.answer), REGISTRY)
    is_correct = result.get("correct", False)
    brier = calibration_aware_score(is_correct, conf)
    return {
        "item_id": item_id,
        "is_correct": is_correct,
        "brier_score": round(brier, 4),
        "confidence": round(conf, 4),
        "model_answer": str(resp.answer),
        "mechanism": mechanism_primary,
    }

# Build evaluation DataFrame
cal_eval_df = pd.DataFrame([{
    "item_id": it["item_id"],
    "question": it["question"],
    "gold_answer": it["gold_answer"],
    "mechanism_primary": it.get("mechanism_primary", ""),
} for it in cal_clean])

# Run sweep with n_jobs=8
cal_results = defaultdict(dict)

for mn, model_obj in verified.items():
    print(f"\n{'='*50}\n  CALIBRATION: {short_name(mn)}\n{'='*50}")
    with kbench.client.enable_cache():
        runs = _nar_cal_item.evaluate(
            llm=[model_obj],
            evaluation_data=cal_eval_df,
            n_jobs=8,
            remove_run_files=True,
        )
    for r in runs:
        if r.result is not None:
            d = r.result
            cal_results[mn][d["item_id"]] = {
                "is_correct": d["is_correct"],
                "brier_score": d["brier_score"],
                "confidence": d["confidence"],
                "model_answer": d["model_answer"],
                "mechanism": d["mechanism"],
            }

    n_correct = sum(v["is_correct"] for v in cal_results[mn].values())
    mean_brier = np.mean([v["brier_score"] for v in cal_results[mn].values()])
    print(f"  Done: {n_correct}/{len(cal_clean)} correct, "
          f"mean 1-Brier = {mean_brier:.4f}")

print(f"\nCalibration sweep complete: {len(cal_results)} models")

In [ ]:
# Cell 6 — Family B 5-Model Sweep

import json as _json

FB_PROMPT = (
    "You are completing a metacognition evaluation task.\n\n"
    "Question:\n{question}\n\n"
    "Choose exactly one action:\n"
    '- "answer": Provide your best answer\n'
    '- "clarify": Ask one specific clarifying question\n'
    '- "verify": Request external verification\n'
    '- "abstain": Decline to answer (genuinely unanswerable)\n\n'
    "Return valid structured output with keys: "
    "decision, answer, confidence, clarification_request, "
    "verification_target, abstention_reason"
)

VALID_ACTIONS = {"answer", "clarify", "verify", "abstain"}
_FB_META = {it["item_id"]: it for it in fb_clean}

def _safe_prompt(llm, prompt):
    """Try structured output first, fall back to unstructured + manual parse."""
    try:
        return llm.prompt(prompt, schema=AbstentionResponse)
    except Exception:
        raw = llm.prompt(prompt)
        d = {}
        if isinstance(raw, dict):
            d = raw
        elif hasattr(raw, '__dict__'):
            d = {k: v for k, v in raw.__dict__.items() if not k.startswith('_')}
        else:
            try:
                d = _json.loads(str(raw))
            except (ValueError, TypeError):
                d = {}
        return AbstentionResponse(**d)

# --- Sub-task for parallel evaluation ---
@kbench.task(name="_nar_fb_item", store_task=False)
def _nar_fb_item(llm, item_id: str, question: str,
                 gold_answer: str, gold_action: str) -> dict:
    with chats.new():
        prompt = FB_PROMPT.format(question=question)
        resp = _safe_prompt(llm, prompt)

    # Normalise confidence
    raw_conf = float(resp.confidence) if resp.confidence is not None else 0.5
    if raw_conf > 1.0:
        raw_conf = raw_conf / 10.0 if raw_conf <= 10.0 else raw_conf / 100.0
    conf = max(0.0, min(1.0, raw_conf))

    # Normalise decision
    decision = str(resp.decision).strip().lower()
    if decision not in VALID_ACTIONS:
        decision = "abstain"

    # Grade answer correctness
    is_correct = False
    if decision == "answer" and resp.answer:
        is_correct = grade_item(item_id, str(resp.answer), REGISTRY,
                                gold_answer=gold_answer).get("correct", False)

    # Score via full v2 scorer
    meta = _FB_META.get(item_id, {})
    utility = score_family_b_item_v2(
        model_decision=decision,
        model_answer=str(resp.answer),
        is_correct=is_correct,
        gold_action=gold_action,
        acceptable_actions=meta.get("acceptable_actions", [gold_action]),
        is_false_presupposition=meta.get("is_false_presupposition", False),
    )

    return {
        "item_id": item_id,
        "model_decision": decision,
        "gold_action": gold_action,
        "utility": round(utility, 4),
        "confidence": round(conf, 4),
        "is_correct": is_correct,
        "model_answer": str(resp.answer),
    }

# Build evaluation DataFrame
fb_eval_df = pd.DataFrame([{
    "item_id": it["item_id"],
    "question": it["question"],
    "gold_answer": it["gold_answer"],
    "gold_action": it["gold_action"],
} for it in fb_clean])

# Run sweep with n_jobs=8
fb_results = defaultdict(dict)

for mn, model_obj in verified.items():
    print(f"\n{'='*50}\n  FAMILY B: {short_name(mn)}\n{'='*50}")
    with kbench.client.enable_cache():
        runs = _nar_fb_item.evaluate(
            llm=[model_obj],
            evaluation_data=fb_eval_df,
            n_jobs=8,
            remove_run_files=True,
        )
    for r in runs:
        if r.result is not None:
            d = r.result
            fb_results[mn][d["item_id"]] = {
                "model_decision": d["model_decision"],
                "gold_action": d["gold_action"],
                "utility": d["utility"],
                "confidence": d["confidence"],
                "is_correct": d["is_correct"],
                "model_answer": d["model_answer"],
            }

    n_items = len(fb_results[mn])
    mean_util = np.mean([v["utility"] for v in fb_results[mn].values()])
    print(f"  Done: {n_items} items, mean utility = {mean_util:+.4f}")

print(f"\nFamily B sweep complete: {len(fb_results)} models")

## Self-Correction Sweep (Family C)

Each model receives 55 items (30 C1 intrinsic, 25 C2 evidence-assisted). For each item:
- **Turn 1:** Model answers the question with ANSWER + CONFIDENCE format
- **Turn 2:** Model receives a review prompt (C1: third-person/fallback; C2: reviewer's note with evidence) and may revise

Transitions are classified as right→right, wrong→right, right→wrong, or wrong→wrong.

In [ ]:
# Cell 6b — Family C Self-Correction 5-Model Sweep

fc_results = defaultdict(dict)

if KAGGLE_ENV and fc_clean:
    @kbench.task(name="_nar_fc_item", store_task=False)
    def _nar_fc_item(llm, item_id, question, gold_answer, subfamily,
                     evidence_snippet, normative_t2_action) -> dict:
        with chats.new():
            # Turn 1
            t1_resp = llm.prompt(question + T1_SUFFIX)
            t1_text = str(t1_resp)
            t1_answer, t1_conf = parse_answer_confidence(t1_text)

            # Turn 2
            if subfamily == "C1":
                t2_prompt = (C1_T2_PRIMARY if len(t1_text) > C1_PRIMARY_MIN_LENGTH
                             else C1_T2_FALLBACK)
            else:  # C2
                t2_prompt = C2_T2_TEMPLATE.format(
                    evidence=evidence_snippet or "")
            t2_resp = llm.prompt(t2_prompt)
            t2_text = str(t2_resp)
            t2_answer, t2_conf = parse_answer_confidence(t2_text)

        # Resolve T2 answer (handle confirmation-without-restatement)
        gold = fc_answer_key.get(item_id, {}).get("gold_answer", "")
        t2_answer_resolved = resolve_t2_answer(t2_answer, t1_answer, gold)

        # Grade both turns
        t1_correct = grade_item(item_id, t1_answer, REGISTRY).get("correct", False)
        t2_correct = grade_item(item_id, t2_answer_resolved, REGISTRY).get("correct", False)

        # Classify transition
        revised = t1_answer.strip().lower() != t2_answer.strip().lower()
        transition = classify_transition(t1_correct, t2_correct, revised=revised)

        # Edit distance
        edit_sim = compute_edit_similarity(t1_text, t2_text)

        return {
            "item_id": item_id,
            "subfamily": subfamily,
            "t1_correct": t1_correct,
            "t2_correct": t2_correct,
            "t1_confidence": t1_conf,
            "t2_confidence": t2_conf,
            "transition": transition,
            "t1_t2_similarity": round(edit_sim, 4),
            "t1_answer": t1_answer[:200],
            "t2_answer": t2_answer[:200],
            "normative_t2_action": normative_t2_action,
        }

    # Build evaluation DataFrame
    fc_eval_df = pd.DataFrame([{
        "item_id": it["item_id"],
        "question": it["question"],
        "gold_answer": it["gold_answer"],
        "subfamily": it["subfamily"],
        "evidence_snippet": it.get("evidence_snippet", ""),
        "normative_t2_action": it.get("normative_t2_action", ""),
    } for it in fc_clean])

    # Run sweep
    for mn, model_obj in verified.items():
        print(f"\n{'='*50}\n  FAMILY C: {short_name(mn)}\n{'='*50}")
        with kbench.client.enable_cache():
            runs = _nar_fc_item.evaluate(
                llm=[model_obj],
                evaluation_data=fc_eval_df,
                n_jobs=4,  # Lower parallelism for multi-turn
                remove_run_files=True,
            )
        for r in runs:
            if r.result is not None:
                fc_results[mn][r.result["item_id"]] = r.result

        n_t1 = sum(v["t1_correct"] for v in fc_results[mn].values())
        n_t2 = sum(v["t2_correct"] for v in fc_results[mn].values())
        n_ran = len(fc_results[mn])
        print(f"  T1: {n_t1}/{n_ran}, T2: {n_t2}/{n_ran}")
else:
    if not fc_clean:
        print("Family C: no items loaded (v2 dataset) -- skipping sweep")
    else:
        print("Family C: no Kaggle SDK -- skipping sweep")

In [ ]:
# Cell 7 — Export Raw Results to CSV (for reproducibility)
import csv

# Calibration audit CSV
cal_rows = []
for mn, items in cal_results.items():
    for iid, v in items.items():
        gold = cal_answer_key.get(iid, {})
        cal_rows.append({
            "model_name": mn, "item_id": iid,
            "question": gold.get("question", ""),
            "gold_answer": gold.get("gold_answer", ""),
            "mechanism": v["mechanism"],
            "model_answer": v["model_answer"],
            "confidence": v["confidence"],
            "is_correct": v["is_correct"],
            "brier_score": round(v["brier_score"], 4),
        })

cal_csv_path = os.path.join(OUTPUT_DIR, "calibration_item_audit.csv")
with open(cal_csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(cal_rows[0].keys()))
    w.writeheader()
    w.writerows(cal_rows)

# Family B audit CSV
fb_rows = []
for mn, items in fb_results.items():
    for iid, v in items.items():
        gold = fb_answer_key.get(iid, {})
        fb_rows.append({
            "model_name": mn, "item_id": iid,
            "question": gold.get("question", ""),
            "gold_answer": gold.get("gold_answer", ""),
            "gold_action": v["gold_action"],
            "model_decision": v["model_decision"],
            "model_answer": v["model_answer"],
            "confidence": v["confidence"],
            "is_correct": v["is_correct"],
            "utility": round(v["utility"], 4),
        })

fb_csv_path = os.path.join(OUTPUT_DIR, "family_b_item_audit.csv")
with open(fb_csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(fb_rows[0].keys()))
    w.writeheader()
    w.writerows(fb_rows)

print(f"Exported: {cal_csv_path} ({len(cal_rows)} rows)")
print(f"Exported: {fb_csv_path} ({len(fb_rows)} rows)")


# Family C audit CSV
fc_rows = []
for mn, items in fc_results.items():
    for iid, v in items.items():
        gold = fc_answer_key.get(iid, {})
        fc_rows.append({
            "model_name": mn, "item_id": iid,
            "subfamily": v["subfamily"],
            "question": gold.get("question", ""),
            "gold_answer": gold.get("gold_answer", ""),
            "t1_answer": v["t1_answer"],
            "t2_answer": v["t2_answer"],
            "t1_correct": v["t1_correct"],
            "t2_correct": v["t2_correct"],
            "t1_confidence": v["t1_confidence"],
            "t2_confidence": v["t2_confidence"],
            "transition": v["transition"],
            "t1_t2_similarity": v["t1_t2_similarity"],
        })

if fc_rows:
    fc_csv_path = os.path.join(OUTPUT_DIR, "family_c_item_audit.csv")
    with open(fc_csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(fc_rows[0].keys()))
        w.writeheader()
        w.writerows(fc_rows)
    print(f"Exported: {fc_csv_path} ({len(fc_rows)} rows)")
else:
    print("Family C: no results to export")

---

## Results

### Calibration Leaderboard (Family A, Clean Set)

In [ ]:
# Cell 9 — Calibration Leaderboard + Reliability Diagram
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.1)

model_order = sorted(cal_results.keys(), key=short_name)
colors = sns.color_palette("tab10", n_colors=len(model_order))
model_colors = {m: colors[i] for i, m in enumerate(model_order)}

# Compute per-model metrics
cal_metrics = {}
for mn in model_order:
    items = cal_results[mn]
    confs = [v["confidence"] for v in items.values()]
    corrects = [v["is_correct"] for v in items.values()]
    briers = [v["brier_score"] for v in items.values()]
    cal_metrics[mn] = {
        "accuracy": float(np.mean(corrects)),
        "mean_1_brier": float(np.mean(briers)),
        "ece": expected_calibration_error(confs, corrects),
        "overconf_rate": overconfidence_rate(confs, corrects),
        "n": len(items),
    }

# Print leaderboard
rows = []
for mn in sorted(cal_metrics, key=lambda m: -cal_metrics[m]["mean_1_brier"]):
    m = cal_metrics[mn]
    rows.append({
        "Model": short_name(mn), "Accuracy": f"{m['accuracy']:.3f}",
        "1-Brier": f"{m['mean_1_brier']:.4f}", "ECE": f"{m['ece']:.4f}",
        "Overconf": f"{m['overconf_rate']:.0%}", "n": m["n"],
    })
print("=== Calibration Leaderboard (Clean Set) ===")
print(pd.DataFrame(rows).to_string(index=False))

# --- Figure 1: Reliability Diagram ---
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect calibration")
for mn in model_order:
    confs = [v["confidence"] for v in cal_results[mn].values()]
    corrects = [v["is_correct"] for v in cal_results[mn].values()]
    bins = accuracy_by_confidence_bucket(confs, corrects)
    xs = [b[0] for b in bins if b[2] > 0]
    ys = [b[1] for b in bins if b[2] > 0]
    ax.plot(xs, ys, "o-", color=model_colors[mn],
            label=short_name(mn), markersize=6)
ax.set_xlabel("Stated Confidence")
ax.set_ylabel("Observed Accuracy")
ax.set_title("Figure 1: Calibration Reliability Diagram")
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig1_reliability_diagram.png"), dpi=150, bbox_inches="tight")
plt.show()

A well-calibrated model traces the diagonal. Above = underconfidence; below = overconfidence.

---

### Family B Leaderboard (Selective Abstention, Clean Set)

In [ ]:
# Cell 11 — Family B Leaderboard + Action Distribution (Figure 2)

fb_metrics = {}
fb_model_order = []
for mn in model_order:
    items = fb_results.get(mn, {})
    if len(items) < 10:
        print(f"  {short_name(mn)}: only {len(items)} FB items -- excluded")
        continue
    fb_model_order.append(mn)
    utils = [v["utility"] for v in items.values()]
    decisions = [v["model_decision"] for v in items.values()]
    golds = [v["gold_action"] for v in items.values()]
    fb_metrics[mn] = {
        "uwaa": compute_uwaa(utils),
        "mean_utility": float(np.mean(utils)),
        "action_accuracy": float(np.mean([d == g for d, g in zip(decisions, golds)])),
        "n": len(items),
    }

# Print leaderboard
rows = []
for mn in sorted(fb_metrics, key=lambda m: -fb_metrics[m]["uwaa"]):
    m = fb_metrics[mn]
    rows.append({
        "Model": short_name(mn), "UWAA": f"{m['uwaa']:.4f}",
        "Mean Util": f"{m['mean_utility']:+.4f}",
        "Action Acc": f"{m['action_accuracy']:.0%}", "n": m["n"],
    })
print("=== Family B Leaderboard (Clean Set) ===")
print(pd.DataFrame(rows).to_string(index=False))

# --- Figure 2: Action Distribution ---
actions = ["answer", "clarify", "verify", "abstain"]
action_colors = {"answer": "#4CAF50", "clarify": "#2196F3",
                 "verify": "#FF9800", "abstain": "#F44336"}

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(fb_model_order))
for action in actions:
    vals = []
    for mn in fb_model_order:
        decisions = [v["model_decision"] for v in fb_results[mn].values()]
        vals.append(sum(1 for d in decisions if d == action) / len(decisions))
    ax.bar([short_name(m) for m in fb_model_order], vals,
           bottom=bottom, label=action, color=action_colors[action])
    bottom += np.array(vals)
ax.set_ylabel("Proportion")
ax.set_title("Figure 2: Action Distribution by Model (Family B)")
ax.legend()
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig2_action_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

### Family C Leaderboard (Self-Correction)

Each model's T1 (initial) and T2 (post-review) accuracy, transition counts, and self-correction rate.

In [ ]:
# Cell 13b — Family C Leaderboard + Transition Summary

if fc_results:
    fc_model_order = sorted(fc_results.keys(), key=short_name)

    # Per-model metrics
    fc_metrics = {}
    for mn in fc_model_order:
        items = fc_results[mn]
        n = len(items)
        t1_correct = sum(v["t1_correct"] for v in items.values())
        t2_correct = sum(v["t2_correct"] for v in items.values())

        # Transition counts
        transitions = defaultdict(int)
        for v in items.values():
            t1c, t2c = v["t1_correct"], v["t2_correct"]
            if t1c and t2c:
                transitions["R->R"] += 1
            elif not t1c and t2c:
                transitions["W->R"] += 1
            elif t1c and not t2c:
                transitions["R->W"] += 1
            else:
                transitions["W->W"] += 1

        wr = transitions["W->R"]
        rw = transitions["R->W"]
        ww = transitions["W->W"]
        t1_wrong = wr + ww
        sc_rate = wr / t1_wrong if t1_wrong > 0 else float("nan")
        t1_right = transitions["R->R"] + rw
        dmg_rate = rw / t1_right if t1_right > 0 else float("nan")

        fc_metrics[mn] = {
            "n": n,
            "t1_acc": t1_correct / n,
            "t2_acc": t2_correct / n,
            "delta": (t2_correct - t1_correct) / n,
            "wr": wr, "rw": rw,
            "sc_rate": sc_rate,
            "dmg_rate": dmg_rate,
            "transitions": dict(transitions),
        }

    # Leaderboard
    rows = []
    for mn in sorted(fc_metrics, key=lambda m: -fc_metrics[m]["delta"]):
        m = fc_metrics[mn]
        rows.append({
            "Model": short_name(mn),
            "T1 Acc": f"{m['t1_acc']:.1%}",
            "T2 Acc": f"{m['t2_acc']:.1%}",
            "Delta": f"{m['delta']:+.1%}",
            "W->R": m["wr"],
            "R->W": m["rw"],
            "SC Rate": f"{m['sc_rate']:.0%}" if not np.isnan(m["sc_rate"]) else "N/A",
            "Dmg Rate": f"{m['dmg_rate']:.1%}" if not np.isnan(m["dmg_rate"]) else "N/A",
            "n": m["n"],
        })
    print("=== Family C Leaderboard ===")
    print(pd.DataFrame(rows).to_string(index=False))

    # Transition matrices
    print("\n=== Transition Matrices ===")
    for mn in fc_model_order:
        m = fc_metrics[mn]
        t = m["transitions"]
        print(f"\n  {short_name(mn)} (n={m['n']}):")
        print(f"    R->R: {t.get('R->R',0):3d}  R->W: {t.get('R->W',0):3d}")
        print(f"    W->R: {t.get('W->R',0):3d}  W->W: {t.get('W->W',0):3d}")
else:
    print("Family C: no results -- skipping leaderboard")

In [ ]:
# Cell 13c — Family C Figures (T2-T1 Delta Bar Chart + Stratum Heatmap)
import matplotlib.pyplot as plt
import seaborn as sns

if fc_results and fc_metrics:
    # --- Figure 5: T2-T1 Accuracy Delta with Bootstrap CIs ---
    fig, ax = plt.subplots(figsize=(8, 5))
    names = [short_name(mn) for mn in fc_model_order]
    deltas = [fc_metrics[mn]["delta"] * 100 for mn in fc_model_order]

    # Bootstrap CIs for delta
    ci_lo, ci_hi = [], []
    for mn in fc_model_order:
        items = list(fc_results[mn].values())
        t1 = np.array([v["t1_correct"] for v in items])
        t2 = np.array([v["t2_correct"] for v in items])
        rng = np.random.default_rng(42)
        boot_deltas = []
        for _ in range(10000):
            idx = rng.integers(0, len(t1), len(t1))
            boot_deltas.append((t2[idx].mean() - t1[idx].mean()) * 100)
        ci_lo.append(np.percentile(boot_deltas, 2.5))
        ci_hi.append(np.percentile(boot_deltas, 97.5))

    errs = [[d - lo for d, lo in zip(deltas, ci_lo)],
            [hi - d for d, hi in zip(deltas, ci_hi)]]

    colors_fc = ["#4CAF50" if d > 0 else "#F44336" for d in deltas]
    ax.barh(names, deltas, xerr=errs, color=colors_fc, edgecolor="white",
            capsize=4, height=0.5)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel("T2 - T1 Accuracy Delta (percentage points)")
    ax.set_title("Figure 5: Self-Correction Accuracy Delta (Family C)")
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig5_fc_delta_bar.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

    # --- Figure 6: Stratum x Model T2 Accuracy Heatmap ---
    strata = sorted(set(
        fc_answer_key.get(v["item_id"], {}).get("stratum", "unknown")
        for items in fc_results.values() for v in items.values()
    ))

    heatmap_data = []
    for stratum in strata:
        row = {"Stratum": stratum}
        for mn in fc_model_order:
            items = [v for v in fc_results[mn].values()
                     if fc_answer_key.get(v["item_id"], {}).get("stratum") == stratum]
            if items:
                row[short_name(mn)] = float(np.mean([v["t2_correct"] for v in items]))
            else:
                row[short_name(mn)] = float("nan")
        row["n"] = sum(1 for v in fc_results[fc_model_order[0]].values()
                       if fc_answer_key.get(v["item_id"], {}).get("stratum") == stratum)
        heatmap_data.append(row)

    df_heat = pd.DataFrame(heatmap_data).set_index("Stratum")
    n_col = df_heat.pop("n")

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(df_heat, annot=True, fmt=".2f", cmap="RdYlGn",
                vmin=0, vmax=1, linewidths=0.5, ax=ax)
    ax.set_title("Figure 6: T2 Accuracy by Stratum x Model (Family C)")
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig6_fc_stratum_heatmap.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("\nItems per stratum:", dict(zip(strata, n_col.values)))
else:
    print("Family C: no results -- skipping figures")

In [ ]:
# Cell 13d — Extended Discrimination Analysis (C1/C2 split, Damage, Pairwise CIs, Cross-Family)

from itertools import combinations

# ═══════════════════════════════════════════════════════════════
# 1. Family C: C1 vs C2 Split Leaderboard
# ═══════════════════════════════════════════════════════════════
if fc_results:
    print("=== Family C — C1 vs C2 Split ===\n")
    for sub in ["C1", "C2"]:
        sub_items_all = {iid for iid, it in fc_answer_key.items()
                         if it.get("subfamily") == sub}
        rows = []
        for mn in sorted(fc_results, key=short_name):
            items = {k: v for k, v in fc_results[mn].items() if k in sub_items_all}
            if not items:
                continue
            n = len(items)
            t1c = sum(v["t1_correct"] for v in items.values())
            t2c = sum(v["t2_correct"] for v in items.values())
            delta = (t2c - t1c) / n
            wr = sum(1 for v in items.values()
                     if not v["t1_correct"] and v["t2_correct"])
            rw = sum(1 for v in items.values()
                     if v["t1_correct"] and not v["t2_correct"])
            t1_wrong = sum(1 for v in items.values() if not v["t1_correct"])
            t1_right = sum(1 for v in items.values() if v["t1_correct"])
            sc = f"{wr}/{t1_wrong}" if t1_wrong else "n/a"
            dmg = f"{rw}/{t1_right}" if t1_right else "n/a"
            rows.append({
                "Model": short_name(mn), "n": n,
                "T1 Acc": f"{t1c/n:.3f}", "T2 Acc": f"{t2c/n:.3f}",
                "Delta": f"{delta:+.3f}",
                "W→R": sc, "R→W": dmg,
                "Dmg%": f"{rw/t1_right:.1%}" if t1_right else "n/a",
            })
        print(f"  {sub} ({len(sub_items_all)} items)")
        print(pd.DataFrame(rows).to_string(index=False))
        print()

# ═══════════════════════════════════════════════════════════════
# 2. Damage Rate Ranking (standalone)
# ═══════════════════════════════════════════════════════════════
if fc_results:
    print("=== Damage Rate Ranking ===\n")
    dmg_rows = []
    for mn in sorted(fc_results, key=short_name):
        items = fc_results[mn]
        t1_right = sum(1 for v in items.values() if v["t1_correct"])
        rw = sum(1 for v in items.values()
                 if v["t1_correct"] and not v["t2_correct"])
        rate = rw / t1_right if t1_right else 0
        dmg_rows.append({"Model": short_name(mn), "R→W": rw,
                         "T1 Correct": t1_right, "Damage Rate": rate})
    dmg_rows.sort(key=lambda r: r["Damage Rate"])
    for r in dmg_rows:
        print(f"  {r['Model']:12s}: {r['R→W']}/{r['T1 Correct']} = {r['Damage Rate']:.1%}")
    print()

# ═══════════════════════════════════════════════════════════════
# 3. Pairwise Bootstrap CIs — Family B (UWAA)
# ═══════════════════════════════════════════════════════════════
if fb_results:
    print("=== Pairwise Family B Comparisons (UWAA, n=bootstrap) ===\n")
    fb_item_ids = sorted(
        set.intersection(*[set(fb_results[mn]) for mn in fb_results])
    )
    fb_pair_rows = []
    for m1, m2 in combinations(sorted(fb_results, key=short_name), 2):
        u1 = np.array([fb_results[m1][iid]["utility"] for iid in fb_item_ids])
        u2 = np.array([fb_results[m2][iid]["utility"] for iid in fb_item_ids])
        diffs = u1 - u2
        obs_delta = np.mean(diffs)
        rng = np.random.RandomState(42)
        boot = [np.mean(rng.choice(diffs, size=len(diffs), replace=True))
                for _ in range(10000)]
        ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
        sig = "Yes" if ci_lo > 0 or ci_hi < 0 else "No"
        fb_pair_rows.append({
            "Pair": f"{short_name(m1)} vs {short_name(m2)}",
            "UWAA Delta": f"{obs_delta:+.4f}",
            "95% CI": f"[{ci_lo:+.4f}, {ci_hi:+.4f}]",
            "Sig": sig,
        })
    print(pd.DataFrame(fb_pair_rows).to_string(index=False))
    print()

# ═══════════════════════════════════════════════════════════════
# 4. Pairwise Bootstrap CIs — Family C (T2-T1 delta)
# ═══════════════════════════════════════════════════════════════
if fc_results:
    print("=== Pairwise Family C Comparisons (T2-T1 Delta) ===\n")
    fc_item_ids = sorted(
        set.intersection(*[set(fc_results[mn]) for mn in fc_results])
    )
    fc_pair_rows = []
    for m1, m2 in combinations(sorted(fc_results, key=short_name), 2):
        # Per-item T2-T1 indicators: +1 if T2 improved, -1 if degraded, 0 if same
        d1 = np.array([int(fc_results[m1][iid]["t2_correct"])
                        - int(fc_results[m1][iid]["t1_correct"])
                        for iid in fc_item_ids])
        d2 = np.array([int(fc_results[m2][iid]["t2_correct"])
                        - int(fc_results[m2][iid]["t1_correct"])
                        for iid in fc_item_ids])
        diffs = d1 - d2
        obs_delta = np.mean(diffs)
        rng = np.random.RandomState(42)
        boot = [np.mean(rng.choice(diffs, size=len(diffs), replace=True))
                for _ in range(10000)]
        ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
        sig = "Yes" if ci_lo > 0 or ci_hi < 0 else "No"
        fc_pair_rows.append({
            "Pair": f"{short_name(m1)} vs {short_name(m2)}",
            "Delta Diff": f"{obs_delta:+.4f}",
            "95% CI": f"[{ci_lo:+.4f}, {ci_hi:+.4f}]",
            "Sig": sig,
        })
    print(pd.DataFrame(fc_pair_rows).to_string(index=False))
    print()

# ═══════════════════════════════════════════════════════════════
# 5. Cross-Family Ranking Table
# ═══════════════════════════════════════════════════════════════
if cal_results and fb_results and fc_results:
    all_models = sorted(
        set(cal_results) & set(fb_results) & set(fc_results),
        key=short_name,
    )
    a_sc = {m: np.mean([v["brier_score"] for v in cal_results[m].values()])
            for m in all_models}
    b_sc = {m: (np.mean([v["utility"] for v in fb_results[m].values()]) + 1) / 2
            for m in all_models}
    c_sc = {}
    for m in all_models:
        items = fc_results[m]
        t1c = sum(v["t1_correct"] for v in items.values())
        t2c = sum(v["t2_correct"] for v in items.values())
        c_sc[m] = (t2c - t1c) / len(items)

    rank_a = sorted(all_models, key=lambda m: -a_sc[m])
    rank_b = sorted(all_models, key=lambda m: -b_sc[m])
    rank_c = sorted(all_models, key=lambda m: -c_sc[m])

    print("=== Cross-Family Rankings ===\n")
    print(f"{'Rank':4s} | {'Family A (1-Brier)':25s} | {'Family B (UWAA)':25s} | {'Family C (T2-T1 Δ)':25s}")
    print("-" * 85)
    for i in range(len(all_models)):
        ma, mb, mc = rank_a[i], rank_b[i], rank_c[i]
        print(f"  {i+1}  | {short_name(ma):8s} ({a_sc[ma]:.3f})      "
              f"| {short_name(mb):8s} ({b_sc[mb]:.3f})      "
              f"| {short_name(mc):8s} ({c_sc[mc]:+.3f})")
    print()

    print("=== Rank Divergence ===\n")
    for m in all_models:
        ra = rank_a.index(m) + 1
        rb = rank_b.index(m) + 1
        rc = rank_c.index(m) + 1
        span = max(ra, rb, rc) - min(ra, rb, rc)
        print(f"  {short_name(m):12s}: A={ra}, B={rb}, C={rc}  (span={span})")

---

### Composite MetaScore (Families A+B+C)

**MetaScore = mean(z_A, z_B, z_C)** — equal-weight z-score composite

Each family is z-standardized independently (population SD), then averaged.
Equal weighting is optimal at n=5 models (Dawes 1979; Davis-Stober 2011:
any data-derived differential weighting will overfit).

**Family C caveat:** α ≈ 0.35 (below 0.55 Haberman threshold for standalone value).
A+B predicts true C better than C's observed score. Family C is included in
the composite for profile information but should NOT be reported as a
standalone axis with independent significance claims.

In [ ]:
# Cell 13 — Composite MetaScore (A+B+C, equal-weight z-score)
from scipy import stats as _sp_stats

if cal_results and fb_results and fc_results:
    _all_models = sorted(
        set(cal_results) & set(fb_results) & set(fc_results),
        key=short_name,
    )
    _names = [short_name(m) for m in _all_models]
    _n = len(_all_models)

    _a = np.array([cal_metrics[m]["mean_1_brier"] for m in _all_models])
    _b = np.array([fb_metrics.get(m, {}).get("uwaa", 0) for m in _all_models])

    # Family C: T2-T1 accuracy delta
    _c = np.zeros(_n)
    _c_dmg = np.zeros(_n)
    for idx, m in enumerate(_all_models):
        items = fc_results[m]
        n_items = len(items)
        t1c = sum(v["t1_correct"] for v in items.values())
        t2c = sum(v["t2_correct"] for v in items.values())
        _c[idx] = (t2c - t1c) / n_items
        t1_right = sum(1 for v in items.values() if v["t1_correct"])
        rw = sum(1 for v in items.values()
                 if v["t1_correct"] and not v["t2_correct"])
        _c_dmg[idx] = rw / t1_right if t1_right > 0 else 0

    def _z(x):
        s = x.std(ddof=0)
        return (x - x.mean()) / s if s > 0 else np.zeros_like(x)

    _za, _zb, _zc = _z(_a), _z(_b), _z(_c)
    _meta = (_za + _zb + _zc) / 3
    _meta_ab = (_za + _zb) / 2
    _rank_abc = _sp_stats.rankdata(-_meta, method="ordinal").astype(int)
    _rank_ab = _sp_stats.rankdata(-_meta_ab, method="ordinal").astype(int)

    rows = []
    for i, nm in enumerate(_names):
        shift = int(_rank_ab[i]) - int(_rank_abc[i])
        arrow = f"{'↑' if shift > 0 else '↓'}{abs(shift)}" if shift != 0 else "="
        rows.append({
            "Model": nm,
            "A (1-Brier)": f"{_a[i]:.4f}",
            "B (UWAA)": f"{_b[i]:.4f}",
            "C (T2-T1 Δ)": f"{_c[i]:+.4f}",
            "C Dmg%": f"{_c_dmg[i]:.1%}",
            "z_A": f"{_za[i]:+.3f}",
            "z_B": f"{_zb[i]:+.3f}",
            "z_C": f"{_zc[i]:+.3f}",
            "MetaScore": f"{_meta[i]:+.3f}",
            "Rank": int(_rank_abc[i]),
            "A+B Rank": int(_rank_ab[i]),
            "Shift": arrow,
        })
    rows.sort(key=lambda r: r["Rank"])
    print("=== Composite MetaScore — Equal-Weight Z-Score (Dawes 1979) ===")
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("Composite: insufficient data (need all 3 families)")

In [ ]:
# Cell 14 — Composite Construction Analysis (z-score, Dirichlet, Hasse, Haberman, Friedman)
# Insert AFTER Cell 18 (existing Composite MetaScore) and BEFORE the "---" markdown divider.
# Dependencies: cal_metrics, fb_metrics, fc_results, model_order, short_name, np, pd (all from earlier cells)

from itertools import permutations as _perms, combinations as _combs
from scipy import stats as _stats

# ═══════════════════════════════════════════════════════════════════════
# Gather per-model scores from already-computed metrics dicts
# ═══════════════════════════════════════════════════════════════════════

_all_models = sorted(
    set(cal_results) & set(fb_results) & set(fc_results),
    key=short_name,
)
_names = [short_name(m) for m in _all_models]
_n = len(_all_models)

_a = np.array([cal_metrics[m]["mean_1_brier"] for m in _all_models])
_b = np.array([fb_metrics[m]["uwaa"] for m in _all_models])

# Family C: T2-T1 accuracy delta
_c = np.zeros(_n)
_c_dmg = np.zeros(_n)
for idx, m in enumerate(_all_models):
    items = fc_results[m]
    n_items = len(items)
    t1c = sum(v["t1_correct"] for v in items.values())
    t2c = sum(v["t2_correct"] for v in items.values())
    _c[idx] = (t2c - t1c) / n_items
    t1_right = sum(1 for v in items.values() if v["t1_correct"])
    rw = sum(1 for v in items.values()
             if v["t1_correct"] and not v["t2_correct"])
    _c_dmg[idx] = rw / t1_right if t1_right > 0 else 0

def _z(x):
    s = x.std(ddof=0)
    return (x - x.mean()) / s if s > 0 else np.zeros_like(x)

_za, _zb, _zc = _z(_a), _z(_b), _z(_c)

# ═══════════════════════════════════════════════════════════════════════
# 1. EQUAL-WEIGHT Z-SCORE COMPOSITE (Dawes-optimal)
# ═══════════════════════════════════════════════════════════════════════
print("=" * 75)
print("COMPOSITE ANALYSIS — Equal-Weight Z-Score MetaScore")
print("=" * 75)

_meta = (_za + _zb + _zc) / 3
_meta_ab = (_za + _zb) / 2
_rank_abc = _stats.rankdata(-_meta, method="ordinal").astype(int)
_rank_ab = _stats.rankdata(-_meta_ab, method="ordinal").astype(int)
_rank_a = _stats.rankdata(-_a, method="ordinal").astype(int)
_rank_b = _stats.rankdata(-_b, method="ordinal").astype(int)
_rank_c = _stats.rankdata(-_c, method="ordinal").astype(int)

rows = []
for i, nm in enumerate(_names):
    shift = int(_rank_ab[i]) - int(_rank_abc[i])
    arrow = f"{'↑' if shift > 0 else '↓'}{abs(shift)}" if shift != 0 else "="
    rows.append({
        "Model": nm,
        "A (1-Brier)": f"{_a[i]:.4f}",
        "B (UWAA)": f"{_b[i]:.4f}",
        "C (T2-T1 Δ)": f"{_c[i]:+.4f}",
        "C Dmg%": f"{_c_dmg[i]:.1%}",
        "z_A": f"{_za[i]:+.3f}",
        "z_B": f"{_zb[i]:+.3f}",
        "z_C": f"{_zc[i]:+.3f}",
        "MetaScore": f"{_meta[i]:+.3f}",
        "Rank": int(_rank_abc[i]),
        "A+B Rank": int(_rank_ab[i]),
        "Shift": arrow,
    })
rows.sort(key=lambda r: r["Rank"])
print(pd.DataFrame(rows).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════
# 2. HABERMAN PRMSE — Does Family C add standalone value?
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 75}")
print("HABERMAN PRMSE — Family C Subscore Value Test")
print("=" * 75)

# Reliability estimates (rough)
_rel_a, _rel_b, _rel_c = 0.88, 0.72, 0.35

_prmse_c = _rel_c

_composite_ab = (_za + _zb) / 2
_r_cX = np.corrcoef(_c, _composite_ab)[0, 1]
_r_ab = np.corrcoef(_a, _b)[0, 1]
_rel_ab = (_rel_a + _rel_b + 2 * _r_ab) / (2 + 2 * _r_ab)
_r_trueC_X = _r_cX / np.sqrt(_rel_c * _rel_ab) if _rel_c * _rel_ab > 0 else 0
_prmse_total = min(_r_trueC_X ** 2 * _rel_ab, 1.0)

print(f"\n  PRMSE(C observed)   = {_prmse_c:.3f}")
print(f"  PRMSE(C from A+B)  = {_prmse_total:.3f}")
if _prmse_c > _prmse_total:
    print(f"  → Family C HAS standalone added value")
else:
    print(f"  → Family C observed score has NO standalone value (A+B predicts true C better)")
    print(f"  → Include in composite for profile information; do not report as standalone axis")

print(f"\n  Threshold: C needs reliability ≥ 0.55 for standalone value")
print(f"  Current estimate: α_C ≈ {_rel_c}")

# ═══════════════════════════════════════════════════════════════════════
# 3. DIRICHLET PAIRWISE STABILITY
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 75}")
print("DIRICHLET PAIRWISE STABILITY (10,000 samples, α=1)")
print("=" * 75)

np.random.seed(42)
_N_SAMP = 10000
_weights = np.random.dirichlet([1.0] * 3, size=_N_SAMP)

_pw = np.zeros((_n, _n))
_rk_counts = np.zeros((_n, _n))

for _w in _weights:
    _comp = _w[0] * _za + _w[1] * _zb + _w[2] * _zc
    _rk = _stats.rankdata(-_comp, method="ordinal").astype(int)
    for i in range(_n):
        _rk_counts[i][_rk[i] - 1] += 1
        for j in range(i + 1, _n):
            if _comp[i] > _comp[j]:
                _pw[i][j] += 1
            else:
                _pw[j][i] += 1

_pw /= _N_SAMP
_rk_prob = _rk_counts / _N_SAMP

# Pairwise table
print(f"\n  P(row beats col) across all weight perturbations:\n")
hdr = "           " + "".join(f" {nm:>8s}" for nm in _names)
print(hdr)
for i, nm in enumerate(_names):
    row_str = f"  {nm:9s}"
    for j in range(_n):
        if i == j:
            row_str += f" {'—':>8s}"
        else:
            row_str += f" {_pw[i][j]:>7.1%}"
    print(row_str)

# Rank probability matrix
print(f"\n  Rank probability matrix:")
print(f"  {'':10s}" + "".join(f" {'Rank '+str(k+1):>8s}" for k in range(_n)) + "  E[rank]")
for i, nm in enumerate(_names):
    row_str = f"  {nm:10s}"
    for k in range(_n):
        row_str += f" {_rk_prob[i][k]:>7.1%}"
    e_rank = sum((k + 1) * _rk_prob[i][k] for k in range(_n))
    row_str += f"  {e_rank:.2f}"
    print(row_str)

# Classify orderings
_robust, _fragile = [], []
for i in range(_n):
    for j in range(i + 1, _n):
        p = max(_pw[i][j], _pw[j][i])
        winner = _names[i] if _pw[i][j] > 0.5 else _names[j]
        loser = _names[j] if winner == _names[i] else _names[i]
        if p >= 0.95:
            _robust.append(f"{winner} > {loser} ({p:.0%})")
        elif p < 0.75:
            _fragile.append(f"{winner} > {loser} ({p:.0%})")

print(f"\n  Robust orderings (P≥95%): {_robust if _robust else 'NONE'}")
print(f"  Fragile orderings (P<75%): {_fragile if _fragile else 'NONE'}")

# Kendall's W
_all_rks = np.array([
    _stats.rankdata(-(_w[0]*_za + _w[1]*_zb + _w[2]*_zc), method="ordinal")
    for _w in _weights
])
_S = np.sum(((_all_rks.sum(axis=0) - _all_rks.sum(axis=0).mean()) ** 2))
_W = 12 * _S / (_N_SAMP ** 2 * (_n ** 3 - _n))
print(f"\n  Kendall's W = {_W:.3f} (1.0 = perfect agreement across weight perturbations)")

# ═══════════════════════════════════════════════════════════════════════
# 4. HASSE PARTIAL ORDER + LINEAR EXTENSION RANK PROBABILITIES
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 75}")
print("HASSE PARTIAL ORDER (Pareto dominance across all 3 families)")
print("=" * 75)

_scores_mat = np.column_stack([_a, _b, _c])

_dominance = set()
for i in range(_n):
    for j in range(_n):
        if i != j:
            if (all(_scores_mat[i, k] >= _scores_mat[j, k] for k in range(3))
                    and any(_scores_mat[i, k] > _scores_mat[j, k] for k in range(3))):
                _dominance.add((i, j))

print(f"\n  Dominance relations (X beats Y on ALL 3 families):")
if _dominance:
    for (i, j) in sorted(_dominance):
        print(f"    {_names[i]} > {_names[j]}")
else:
    print(f"    NONE")

print(f"\n  Incomparable pairs:")
for i in range(_n):
    for j in range(i + 1, _n):
        if (i, j) not in _dominance and (j, i) not in _dominance:
            i_wins = [["A", "B", "C"][k] for k in range(3) if _scores_mat[i, k] > _scores_mat[j, k]]
            j_wins = [["A", "B", "C"][k] for k in range(3) if _scores_mat[j, k] > _scores_mat[i, k]]
            print(f"    {_names[i]} ↔ {_names[j]}: "
                  f"{_names[i]} wins {i_wins}, {_names[j]} wins {j_wins}")

# Linear extensions
_consistent = []
for perm in _perms(range(_n)):
    valid = True
    for (i, j) in _dominance:
        if list(perm).index(i) > list(perm).index(j):
            valid = False
            break
    if valid:
        _consistent.append(perm)

_le_rk = np.zeros((_n, _n))
for perm in _consistent:
    for rank, model in enumerate(perm):
        _le_rk[model][rank] += 1
_le_rk /= len(_consistent)

print(f"\n  Linear extensions: {len(_consistent)}/{_n}! = {len(_consistent)}/120 permutations consistent")
print(f"\n  Rank probability matrix (weight-free, from Pareto dominance only):")
print(f"  {'':10s}" + "".join(f" {'Rank '+str(k+1):>8s}" for k in range(_n)) + "  E[rank]")
for i, nm in enumerate(_names):
    row_str = f"  {nm:10s}"
    for k in range(_n):
        row_str += f" {_le_rk[i][k]:>7.1%}"
    e_rank = sum((k + 1) * _le_rk[i][k] for k in range(_n))
    row_str += f"  {e_rank:.2f}"
    print(row_str)

# ═══════════════════════════════════════════════════════════════════════
# 5. KEMENY-YOUNG CONSENSUS RANKING
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 75}")
print("KEMENY-YOUNG CONSENSUS RANKING")
print("=" * 75)

def _kt_dist(r1, r2):
    d = 0
    for i in range(len(r1)):
        for j in range(i + 1, len(r1)):
            if (r1[i] - r1[j]) * (r2[i] - r2[j]) < 0:
                d += 1
    return d

_input_rks = [
    _stats.rankdata(-_a, method="ordinal").astype(int),
    _stats.rankdata(-_b, method="ordinal").astype(int),
    _stats.rankdata(-_c, method="ordinal").astype(int),
]

_best_d = float("inf")
_best_p = []
for perm in _perms(range(1, _n + 1)):
    cand = np.array(perm)
    td = sum(_kt_dist(cand, r) for r in _input_rks)
    if td < _best_d:
        _best_d = td
        _best_p = [cand]
    elif td == _best_d:
        _best_p.append(cand)

print(f"\n  Min total Kendall-tau distance: {_best_d} (of max {3 * _n * (_n-1) // 2})")
for bp in _best_p:
    ordered = [_names[i] for i in np.argsort(bp)]
    print(f"  Consensus: {' > '.join(ordered)}")
for fam, r in zip(["A", "B", "C"], _input_rks):
    print(f"    Distance from Family {fam}: {_kt_dist(_best_p[0], r)}")

# ═══════════════════════════════════════════════════════════════════════
# 6. FRIEDMAN + NEMENYI CRITICAL DIFFERENCE
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 75}")
print("FRIEDMAN TEST + NEMENYI CRITICAL DIFFERENCE")
print("=" * 75)

_avg_rk = np.mean([r.astype(float) for r in _input_rks], axis=0)
N_fam = 3
k_mod = _n
_chi2 = 12 * N_fam / (k_mod * (k_mod + 1)) * (
    np.sum(_avg_rk ** 2) - k_mod * (k_mod + 1) ** 2 / 4
)
_F_id = (N_fam - 1) * _chi2 / (N_fam * (k_mod - 1) - _chi2) if (N_fam * (k_mod - 1) - _chi2) > 0 else 0
_p_fried = 1 - _stats.f.cdf(_F_id, k_mod - 1, (k_mod - 1) * (N_fam - 1))

# Nemenyi CD: q_α(k=5, α=0.05) ≈ 2.728
_q_alpha = 2.728
_CD = _q_alpha * np.sqrt(k_mod * (k_mod + 1) / (6 * N_fam))

print(f"\n  Average ranks across 3 families:")
for i, nm in enumerate(_names):
    print(f"    {nm:10s}: {_avg_rk[i]:.2f}")
print(f"\n  Friedman χ² = {_chi2:.2f}, Iman-Davenport F = {_F_id:.2f}, p = {_p_fried:.3f}")
print(f"  Significant at α=0.05? {'Yes' if _p_fried < 0.05 else 'No'}")
print(f"\n  Nemenyi CD = {_CD:.2f}  (need |Δrank| > {_CD:.2f} for significance)")
_max_diff = max(abs(_avg_rk[i] - _avg_rk[j]) for i in range(_n) for j in range(i+1, _n))
print(f"  Max observed |Δrank| = {_max_diff:.2f}")
if _max_diff < _CD:
    print(f"  → No model pair is statistically separable with only 3 families")
    print(f"  → This is expected: the benchmark's value is in the PROFILE, not the ranking")

# ═══════════════════════════════════════════════════════════════════════
# 7. SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 75}")
print("SUMMARY: MetaScore Leaderboard with Uncertainty")
print("=" * 75)

_summary = []
for i in range(_n):
    _summary.append({
        "Model": _names[i],
        "MetaScore": f"{_meta[i]:+.3f}",
        "Rank": int(_rank_abc[i]),
        "P(Rank 1)": f"{_rk_prob[i][0]:.0%}",
        "P(Rank 5)": f"{_rk_prob[i][_n-1]:.0%}",
        "95% Rank Set": f"{{{int(np.min(np.where(_rk_prob[i]>0.025)[0])+1)}"
                        f"–{int(np.max(np.where(_rk_prob[i]>0.025)[0])+1)}}}",
        "Dominates": ", ".join(
            _names[j] for j in range(_n) if (i, j) in _dominance
        ) or "—",
    })
_summary.sort(key=lambda r: r["Rank"])
print(f"\n{pd.DataFrame(_summary).to_string(index=False)}")

print(f"\n  Equal-weight z-score composite (Dawes 1979: optimal at n={_n})")
print(f"  Dirichlet stability: W = {_W:.2f} | {len(_robust)} robust, {len(_fragile)} fragile orderings")
print(f"  Haberman: Family C α≈{_rel_c} < 0.55 threshold → include in composite, not standalone")
print(f"  Friedman: p = {_p_fried:.2f} → no significant model separation from 3 families alone")
print(f"\n  Key finding: No single family discriminates all models.")
print(f"  The composite reveals capability PROFILES that individual metrics cannot.")

# ═══════════════════════════════════════════════════════════════════════
# EXPORT for benchmark notebook
# ═══════════════════════════════════════════════════════════════════════
_export_rows = []
for i, m in enumerate(_all_models):
    _export_rows.append({
        "model_name": m,
        "model_short": _names[i],
        "a_score": round(_a[i], 4),
        "b_score": round(_b[i], 4),
        "c_delta": round(_c[i], 4),
        "c_damage_rate": round(_c_dmg[i], 4),
        "z_a": round(_za[i], 4),
        "z_b": round(_zb[i], 4),
        "z_c": round(_zc[i], 4),
        "metascore": round(_meta[i], 4),
        "rank_abc": int(_rank_abc[i]),
        "rank_ab_only": int(_rank_ab[i]),
        "dirichlet_p_rank1": round(_rk_prob[i][0], 3),
        "dirichlet_e_rank": round(sum((k+1)*_rk_prob[i][k] for k in range(_n)), 2),
    })

import csv as _csv
_exp_path = os.path.join(OUTPUT_DIR, "composite_analysis.csv")
with open(_exp_path, "w", newline="") as f:
    w = _csv.DictWriter(f, fieldnames=list(_export_rows[0].keys()))
    w.writeheader()
    w.writerows(_export_rows)
print(f"\n  Exported: {_exp_path}")

---

## Statistical Analysis

Pairwise bootstrap CIs (10,000 resamples, seed=42) with Holm-Bonferroni correction for multiple comparisons.

### Calibration (Family A)

In [ ]:
# Cell 15 — Pairwise Bootstrap CIs + Forest Plot (Figure 3)

models = sorted(cal_results.keys())
common_items = sorted(set.intersection(
    *(set(cal_results[m].keys()) for m in models)
))

pairwise = []
all_p = {}

for m_a, m_b in combinations(models, 2):
    pair = f"{short_name(m_a)} vs {short_name(m_b)}"
    brier_a = [cal_results[m_a][i]["brier_score"] for i in common_items]
    brier_b = [cal_results[m_b][i]["brier_score"] for i in common_items]
    boot = paired_bootstrap_ci(brier_a, brier_b)

    sig = "Yes" if boot["ci_lower"] > 0 or boot["ci_upper"] < 0 else "No"
    pairwise.append({
        "Pair": pair,
        "Delta": boot["observed_diff"],
        "CI_lo": boot["ci_lower"],
        "CI_hi": boot["ci_upper"],
        "Sig": sig,
    })
    # Use whether CI excludes 0 as proxy p-value for Holm
    all_p[pair] = 0.001 if sig == "Yes" else 0.50

corrected = holm_correction(all_p)
n_sig = sum(1 for v in corrected.values() if v["significant_0.05"])

# Print table
rows = []
for pw in pairwise:
    holm_sig = corrected.get(pw["Pair"], {}).get("significant_0.05", False)
    rows.append({
        "Pair": pw["Pair"],
        "Brier Delta": f"{pw['Delta']:+.4f}",
        "95% CI": f"[{pw['CI_lo']:.4f}, {pw['CI_hi']:.4f}]",
        "Sig (Holm)": "Yes" if holm_sig else "No",
    })
print(f"=== Pairwise Calibration Comparisons (n={len(common_items)}) ===")
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nHolm-Bonferroni: {n_sig}/{len(corrected)} significant at alpha=0.05")

# --- Figure 3: Brier Forest Plot ---
fig, ax = plt.subplots(figsize=(10, 5))
pairs = [pw["Pair"] for pw in pairwise]
for i, pw in enumerate(pairwise):
    ax.errorbar(pw["Delta"], i,
                xerr=[[pw["Delta"] - pw["CI_lo"]],
                      [pw["CI_hi"] - pw["Delta"]]],
                fmt="o", color="steelblue", capsize=4, markersize=6)
ax.axvline(0, color="red", linestyle="--", alpha=0.4)
ax.set_yticks(range(len(pairs)))
ax.set_yticklabels(pairs, fontsize=9)
ax.set_xlabel("Brier Score Difference (A - B)")
ax.set_title("Figure 3: Pairwise Brier Differences with 95% Bootstrap CI")
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig3_brier_forest_plot.png"), dpi=150, bbox_inches="tight")
plt.show()


# --- Family C: Pairwise T2-T1 Delta Comparisons ---
if fc_results:
    print("\n=== Family C: Pairwise T2-T1 Delta Comparisons ===")
    fc_models = sorted(fc_results.keys())
    fc_common = sorted(set.intersection(
        *(set(fc_results[m].keys()) for m in fc_models)
    ))

    fc_pairwise = []
    for m_a, m_b in combinations(fc_models, 2):
        pair = f"{short_name(m_a)} vs {short_name(m_b)}"
        # Per-item delta difference: (T2-T1)_a - (T2-T1)_b
        delta_a = [int(fc_results[m_a][i]["t2_correct"]) - int(fc_results[m_a][i]["t1_correct"])
                   for i in fc_common]
        delta_b = [int(fc_results[m_b][i]["t2_correct"]) - int(fc_results[m_b][i]["t1_correct"])
                   for i in fc_common]
        boot = paired_bootstrap_ci(delta_a, delta_b)
        sig = "Yes" if boot["ci_lower"] > 0 or boot["ci_upper"] < 0 else "No"
        fc_pairwise.append({
            "Pair": pair,
            "Delta Diff": f"{boot['observed_diff']:+.4f}",
            "95% CI": f"[{boot['ci_lower']:.4f}, {boot['ci_upper']:.4f}]",
            "Sig": sig,
        })
    print(pd.DataFrame(fc_pairwise).to_string(index=False))

In [ ]:
# Cell 16 — Per-Mechanism Heatmap (Figure 4) + Bridge Analysis

# --- Figure 4: Mechanism x Model Accuracy Heatmap ---
mechanisms = sorted(set(
    v["mechanism"] for items in cal_results.values()
    for v in items.values() if v["mechanism"]
))

mech_data = []
for mech in mechanisms:
    row = {"Mechanism": mech}
    for mn in model_order:
        items = {k: v for k, v in cal_results[mn].items()
                 if v["mechanism"] == mech}
        if items:
            row[short_name(mn)] = float(np.mean(
                [v["is_correct"] for v in items.values()]))
        else:
            row[short_name(mn)] = float("nan")
    row["n"] = sum(1 for v in cal_results[model_order[0]].values()
                   if v["mechanism"] == mech)
    mech_data.append(row)

df_mech = pd.DataFrame(mech_data).set_index("Mechanism")
n_col = df_mech.pop("n")

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df_mech, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Figure 4: Accuracy by Mechanism x Model")
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig4_mechanism_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("\nItems per mechanism:", dict(zip(mechanisms, n_col.values)))

# --- Bridge: Spearman confidence-accuracy correlation ---
print("\n=== Bridge: Confidence-Accuracy Correlation ===")
bridge_rows = []
for mn in model_order:
    confs = [v["confidence"] for v in cal_results[mn].values()]
    corrects = [float(v["is_correct"]) for v in cal_results[mn].values()]
    sp = spearman_with_ci(confs, corrects)
    bridge_rows.append({
        "Model": short_name(mn),
        "Spearman rho": f"{sp['rho']:.3f}",
        "p-value": f"{sp['p_value']:.4f}",
        "95% CI": f"[{sp['ci_lower']:.3f}, {sp['ci_upper']:.3f}]",
    })
print(pd.DataFrame(bridge_rows).to_string(index=False))

---

## What This Benchmark Reveals

1. **Calibration quality differs between models** -- some are systematically overconfident on wrong answers, while others maintain realistic confidence estimates.

2. **Action selection is distinct from answer quality** -- a model that scores well on calibration does not necessarily choose the right metacognitive action (answer vs. abstain vs. verify vs. clarify).

3. **Mechanism-level analysis reveals systematic failure patterns** -- models have characteristic weaknesses (anchoring traps, monitoring traps) that a global score would obscure.

4. **Monitoring and control are partially dissociable** -- the bridge analysis (Spearman rho) measures whether calibration quality predicts action-selection quality. A weak correlation means these are worth measuring separately.

5. **Self-correction ability varies sharply across models** -- some models show genuine error correction (W→R transitions without corresponding R→W damage), while others are rigid or net-damaging. Edit-distance analysis reveals whether revision is targeted or wholesale rewriting.

6. **No single family discriminates all models** -- Family C reshuffles the A+B ranking (e.g., Pro drops from rank 2 to rank 3 due to high damage rate). The composite reveals capability profiles that individual metrics cannot.

7. **Flash and Haiku rankings are robust; the middle three are statistically interchangeable** -- Dirichlet stability analysis shows Flash is rank 1 in 86% of weight perturbations, Haiku is rank 5 in 59%. Pro, Sonnet, and DeepSeek occupy ranks 2-4 with high uncertainty.

8. **Family C has no standalone subscore value** (Haberman PRMSE test: α ≈ 0.35 < 0.55 threshold). It contributes unique profile information to the composite but should not be reported as an independent axis.

---

## Limitations

- **Sample sizes**: 105 calibration items, 72 Family B items, and 51 Family C items (4 unresolved excluded) provide adequate power for moderate effects but limit per-stratum and per-mechanism inference.
- **Single run**: Results reflect one pass per model at temperature=0.0 (Families A/B) or SDK default (Family C). Multi-run variance analysis is planned.
- **Family C reliability**: α ≈ 0.35 is below the 0.55 standalone threshold. A+B predicts true C better than C predicts itself. Include in composite but caveat.
- **Friedman test non-significant**: p = 0.376 with 3 families — insufficient to statistically separate 5 models. The benchmark's value is in the PROFILE, not the ranking.
- **Models evaluated in default configuration**: Tool use is unavailable. Chain-of-thought/reasoning varies by model and is not toggleable. This is by design.
- **Family C multi-turn constraint**: Kaggle SDK may not expose temperature control. Sprint v2 results (temp=0) remain the definitive analysis.

## Next Steps

- Families D-E (grounding sensitivity, control policy adaptation) are designed but not yet instrumented.
- Multi-run replication to estimate test-retest reliability and improve Family C's α.
- Expand Family C item pool (75-150 items needed for defensible SC rate comparisons).
- Apply Mogstad et al. (2024) confidence sets for formal rank inference.